In [4]:
import sys, subprocess

# Step 1: cirq ও qualtran সরাও
subprocess.run([sys.executable, "-m", "pip", "uninstall", 
                "cirq", "qualtran", "pennylane-cirq", "-y"])

# Step 2: kernel restart করুন — এরপর নিচের cell চালান
print("Now RESTART KERNEL, then run next cell")

Now RESTART KERNEL, then run next cell


In [2]:
pip install pennylane 

  Using cached pennylane_lightning-0.45.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.4 MB 3.4 MB/s eta 0:00:02
   ----------------- ---------------------- 2.4/5.4 MB 7.1 MB/s eta 0:00:01
   ---------------------------------- ----- 4.7/5.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 5.4/5.4 MB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/937.5 kB ? eta -:--:--
   ---------------------------------------- 937.5/937.5 kB 6.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.5 MB 9.4 MB/s eta 0:00:02
   ------------- -------------------------- 4.2/12.5 MB 9.9 MB/s eta 0:00:01
   ------------------- -------------------- 6.0/12.5 MB 9.8 MB/s eta 0:00:01
   ------------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pennylane as qml
print(qml.__version__)  # যেকোনো version হোক, এখন import হবে

0.45.0


In [ ]:
"""
Adaptive Quantum Capsule Network (QCN) — sklearn Digits Binary Classification
==============================================================================
TRUE ADAPTIVE DEPTH:
  Primary   → Gradient Variance based  (Barren Plateau detect)
  Secondary → Performance based        (accuracy plateau detect)

Depth বাড়ার condition:
  1. grad_variance[c] < GRAD_VAR_THRESHOLD  (primary)
     OR
  2. accuracy improvement < ACC_IMPROVE_THRESHOLD  (secondary)
  +  minimum DEPTH_ADAPT_COOLDOWN epochs gap between adaptations
  +  maximum MAX_DEPTH cap

Outputs:
  training_log.csv
  grad_variance_log.csv
  depth_adaptation_log.csv   ← NEW: কোন epoch এ কোন capsule এর depth বদলাল
  effective_dimension.csv
  efficiency_analysis.csv
  capsule_correlation.csv
"""

import os
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ═══════════════════════════════════════════════════════════════
# 0.  ALL HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════

DIGIT_A     = 0
DIGIT_B     = 1

N_CAPS      = 4
CAP_SIZE    = 4
DEPTHS      = [2, 2, 2, 2]   # starting depth (low — adaptive বাড়াবে)
MAX_DEPTH   = 6               # কোনো capsule এর depth এর উপরে সীমা

N_EPOCHS    = 30
BATCH_SIZE  = 8
LR          = 0.05

QUICK_EP    = 5
TEST_SIZE   = 0.25
RANDOM_SEED = 42

OUTPUT_DIR  = "C:\\Users\\User\\Downloads\\Barren-Pleatau-Model\\"

# ── Adaptive Depth Hyperparameters ──────────────────────────────
GRAD_VAR_THRESHOLD   = 1e-4   # এর নিচে গেলে → depth বাড়াও (primary)
ACC_IMPROVE_THRESHOLD = 0.005 # এর কম improve হলে → depth বাড়াও (secondary)
DEPTH_ADAPT_COOLDOWN  = 3     # কতো epoch পর পর adapt করা যাবে
ADAPT_CHECK_START     = 5     # কতো epoch পর থেকে adaptation শুরু হবে

# ── Derived constants ────────────────────────────────────────────
N_QUBITS     = N_CAPS * CAP_SIZE
IMG_SIDE     = int(np.sqrt(N_QUBITS))
N_SAMPLES_ED = [100, 500, 1000, 5000, 10000]

assert len(DEPTHS) == N_CAPS,     "len(DEPTHS) must equal N_CAPS"
assert IMG_SIDE ** 2 == N_QUBITS, "N_QUBITS must be a perfect square"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print(f"  TRUE ADAPTIVE QCN Config")
print(f"  Digits         : {DIGIT_A} vs {DIGIT_B}")
print(f"  Capsules       : {N_CAPS}  x  {CAP_SIZE} qubits  =  {N_QUBITS} total")
print(f"  Start Depths   : {DEPTHS}  (max={MAX_DEPTH})")
print(f"  Grad Threshold : {GRAD_VAR_THRESHOLD}")
print(f"  Acc Threshold  : {ACC_IMPROVE_THRESHOLD}")
print(f"  Cooldown       : {DEPTH_ADAPT_COOLDOWN} epochs")
print(f"  Epochs         : {N_EPOCHS}   Batch: {BATCH_SIZE}   LR: {LR}")
print("=" * 60)

# ═══════════════════════════════════════════════════════════════
# 1.  DATASET  +  PCA PREPROCESSING
# ═══════════════════════════════════════════════════════════════
print(f"\n[Data] Loading sklearn digits ({DIGIT_A} vs {DIGIT_B}) ...")

digits = load_digits()
mask   = (digits.target == DIGIT_A) | (digits.target == DIGIT_B)
X_raw  = digits.data[mask]
y_raw  = digits.target[mask]

y = np.where(y_raw == DIGIT_A, -1.0, 1.0)

pca      = PCA(n_components=N_QUBITS, random_state=RANDOM_SEED)
X_pca    = pca.fit_transform(X_raw)

scaler   = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

X_images = X_scaled.reshape(-1, IMG_SIDE, IMG_SIDE)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"  PCA variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"  Train : {len(X_train)}   Test : {len(X_test)}")

# ═══════════════════════════════════════════════════════════════
# 2.  QUANTUM DEVICE
# ═══════════════════════════════════════════════════════════════
dev = qml.device("default.qubit", wires=N_QUBITS)

# ═══════════════════════════════════════════════════════════════
# 3.  CIRCUIT BUILDER
# ═══════════════════════════════════════════════════════════════
def build_circuit(n_qubits, n_caps, cap_size, depths, device):
    offsets = [0]
    for c in range(n_caps):
        offsets.append(offsets[-1] + depths[c] * cap_size * 4)
    total_params = offsets[-1]
    max_d        = max(depths)

    @qml.qnode(device, diff_method="best")
    def circuit(params, x):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)

        for layer in range(max_d):
            for c in range(n_caps):
                if layer >= depths[c]:
                    continue

                w  = list(range(c * cap_size, (c + 1) * cap_size))
                cp = params[offsets[c]: offsets[c + 1]].reshape(
                         depths[c], cap_size, 4)

                xc = [float(x[c * cap_size + j]) for j in range(cap_size)]

                for j, wire in enumerate(w):
                    qml.RY(xc[j] * cp[layer, j, 3], wires=wire)

                for j, wire in enumerate(w):
                    qml.Rot(cp[layer, j, 0],
                            cp[layer, j, 1],
                            cp[layer, j, 2], wires=wire)

                if layer % 2 == 0:
                    for j in range(len(w) - 1):
                        qml.CNOT(wires=[w[j], w[j + 1]])
                    if len(w) > 2:
                        qml.CNOT(wires=[w[-1], w[0]])

            if layer % 2 == 0:
                ic_offset = (layer // 2) % 2
                for c in range(ic_offset, n_caps - 1, 2):
                    if layer < depths[c] and layer < depths[c + 1]:
                        boundary_a = (c + 1) * cap_size - 1
                        boundary_b = (c + 1) * cap_size
                        qml.CNOT(wires=[boundary_a, boundary_b])

        return qml.math.stack([
            qml.expval(qml.PauliZ(c * cap_size)) for c in range(n_caps)
        ])

    return circuit, total_params, offsets


# ═══════════════════════════════════════════════════════════════
# 4.  LOSS + PREDICTION HELPERS
# ═══════════════════════════════════════════════════════════════
def make_helpers(circuit_fn):
    def predict_one(params, x):
        return qml.math.mean(circuit_fn(params, x))

    def mse_loss(params, X_batch, y_batch):
        preds = qml.math.stack([predict_one(params, x) for x in X_batch])
        return qml.math.mean((preds - y_batch) ** 2)

    def accuracy(params, X, y):
        signs = np.array([float(np.sign(predict_one(params, x))) for x in X])
        signs = np.where(signs == 0, 1.0, signs)
        return float(np.mean(signs == y))

    return predict_one, mse_loss, accuracy


# ═══════════════════════════════════════════════════════════════
# 5.  ADAPTIVE DEPTH ENGINE
# ═══════════════════════════════════════════════════════════════
def check_and_adapt_depths(
    depths, params, offsets,
    cap_grad_vars,          # per-capsule gradient variance list
    prev_acc, curr_acc,
    last_adapt_epoch, epoch,
    cap_size
):
    """
    Returns: (new_depths, new_params, new_offsets, adapted_capsules)
    Priority: Gradient Variance (primary) > Accuracy Plateau (secondary)
    """
    if epoch < ADAPT_CHECK_START:
        return depths, params, offsets, []

    if (epoch - last_adapt_epoch) < DEPTH_ADAPT_COOLDOWN:
        return depths, params, offsets, []

    adapted = []
    new_depths = list(depths)

    for c in range(N_CAPS):
        if new_depths[c] >= MAX_DEPTH:
            continue

        grad_trigger = cap_grad_vars[c] < GRAD_VAR_THRESHOLD
        acc_trigger  = (curr_acc - prev_acc) < ACC_IMPROVE_THRESHOLD

        # Primary: gradient → adapt
        # Secondary: accuracy plateau → adapt (only if grad is also low-ish)
        if grad_trigger or (acc_trigger and cap_grad_vars[c] < GRAD_VAR_THRESHOLD * 10):
            reason = "grad_var" if grad_trigger else "acc_plateau"
            new_depths[c] += 1
            adapted.append((c, depths[c], new_depths[c], reason,
                             cap_grad_vars[c]))

    if not adapted:
        return depths, params, offsets, []

    # Rebuild circuit + expand params (new params init near 0)
    new_offsets = [0]
    for c in range(N_CAPS):
        new_offsets.append(new_offsets[-1] + new_depths[c] * cap_size * 4)
    new_total = new_offsets[-1]

    new_params = np.zeros(new_total, requires_grad=True)
    for c in range(N_CAPS):
        old_size = offsets[c + 1] - offsets[c]
        new_params[new_offsets[c]: new_offsets[c] + old_size] = \
            params[offsets[c]: offsets[c + 1]]
        # extra new params → small random init
        extra_start = new_offsets[c] + old_size
        extra_end   = new_offsets[c + 1]
        if extra_end > extra_start:
            new_params[extra_start:extra_end] = \
                np.random.uniform(0, 0.1, extra_end - extra_start)

    return new_depths, new_params, new_offsets, adapted


# ═══════════════════════════════════════════════════════════════
# 6.  INITIAL BUILD
# ═══════════════════════════════════════════════════════════════
current_depths = list(DEPTHS)
circuit, total_params, offsets = build_circuit(
    N_QUBITS, N_CAPS, CAP_SIZE, current_depths, dev
)
predict_one, mse_loss, accuracy = make_helpers(circuit)
print(f"\nCircuit ready  |  total_params = {total_params}  |  depths = {current_depths}")

# ═══════════════════════════════════════════════════════════════
# 7.  TRAINING LOOP  +  ADAPTIVE DEPTH
# ═══════════════════════════════════════════════════════════════
print(f"\n[Train] Starting Adaptive Training ({N_EPOCHS} epochs) ...")

np.random.seed(RANDOM_SEED)
params    = np.random.uniform(0, np.pi, total_params, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=LR)

training_log      = []
grad_var_log      = []
depth_adapt_log   = []

prev_test_acc     = 0.0
last_adapt_epoch  = 0

for epoch in range(1, N_EPOCHS + 1):

    grad_fn = qml.grad(mse_loss)

    perm   = np.random.permutation(len(X_train))
    X_shuf = X_train[perm]
    y_shuf = y_train[perm]

    epoch_loss, n_batches = 0.0, 0

    for start in range(0, len(X_train), BATCH_SIZE):
        Xb = X_shuf[start: start + BATCH_SIZE]
        yb = y_shuf[start: start + BATCH_SIZE]
        params, loss_val = optimizer.step_and_cost(
            lambda p: mse_loss(p, Xb, yb), params
        )
        epoch_loss += float(loss_val)
        n_batches  += 1

    avg_loss  = epoch_loss / n_batches
    train_acc = accuracy(params, X_train, y_train)
    test_acc  = accuracy(params, X_test,  y_test)

    # ── Per-capsule gradient variance ────────────────────────────
    grad_samples = []
    eval_indices = perm[:min(16, len(X_train))]

    for idx in eval_indices:
        g = grad_fn(params,
                    X_train[idx: idx + 1],
                    y_train[idx: idx + 1])
        grad_samples.append(np.array(g[0]).flatten())

    grad_samples    = np.array(grad_samples)           # (16, total_params)
    param_variances = np.var(grad_samples, axis=0)     # (total_params,)

    # per-capsule variance (mean of that capsule's param variances)
    cap_grad_vars = []
    for c in range(N_CAPS):
        start_idx = offsets[c]
        end_idx   = offsets[c + 1]
        cap_grad_vars.append(float(np.mean(param_variances[start_idx:end_idx])))

    epoch_grad_var_mean = float(np.mean(param_variances))
    epoch_grad_var_std  = float(np.std(param_variances))

    # ── Adaptive Depth Check ──────────────────────────────────────
    current_depths, params, offsets, adapted = check_and_adapt_depths(
        current_depths, params, offsets,
        cap_grad_vars,
        prev_test_acc, test_acc,
        last_adapt_epoch, epoch,
        CAP_SIZE
    )

    if adapted:
        # Rebuild circuit + helpers with new depths
        circuit, total_params, offsets = build_circuit(
            N_QUBITS, N_CAPS, CAP_SIZE, current_depths, dev
        )
        predict_one, mse_loss, accuracy = make_helpers(circuit)
        optimizer = qml.AdamOptimizer(stepsize=LR)  # reset optimizer state
        last_adapt_epoch = epoch

        for (c, old_d, new_d, reason, gv) in adapted:
            print(f"  *** ADAPT Epoch {epoch}: Capsule {c}  "
                  f"depth {old_d}→{new_d}  reason={reason}  "
                  f"grad_var={gv:.3e} ***")
            depth_adapt_log.append({
                "Epoch":       epoch,
                "Capsule":     c,
                "Old_Depth":   old_d,
                "New_Depth":   new_d,
                "Reason":      reason,
                "Grad_Var":    f"{gv:.4e}",
                "Test_Acc":    round(test_acc, 4),
            })

    training_log.append({
        "Epoch":         epoch,
        "Loss":          round(avg_loss,  6),
        "Train_Acc":     round(train_acc, 4),
        "Test_Acc":      round(test_acc,  4),
        "Depths":        str(current_depths),
        "Grad_Var_Mean": f"{epoch_grad_var_mean:.4e}",
        "Grad_Var_Std":  f"{epoch_grad_var_std:.4e}",
        "Cap_Grad_Vars": str([f"{v:.3e}" for v in cap_grad_vars]),
    })

    grad_var_log.append({
        "Epoch":         epoch,
        "Grad_Var_Mean": epoch_grad_var_mean,
        "Grad_Var_Std":  epoch_grad_var_std,
    })

    prev_test_acc = test_acc

    print(f"  Epoch {epoch:>3}/{N_EPOCHS}  |  "
          f"loss={avg_loss:.4f}  "
          f"train={train_acc:.3f}  "
          f"test={test_acc:.3f}  "
          f"depths={current_depths}  "
          f"grad_var={epoch_grad_var_mean:.3e}")

# ── Save CSVs ────────────────────────────────────────────────────
pd.DataFrame(training_log).to_csv(
    f"{OUTPUT_DIR}training_log.csv", index=False)
print(f"\n  Saved -> training_log.csv")

pd.DataFrame(grad_var_log).to_csv(
    f"{OUTPUT_DIR}grad_variance_log.csv", index=False)
print(f"  Saved -> grad_variance_log.csv")

pd.DataFrame(depth_adapt_log).to_csv(
    f"{OUTPUT_DIR}depth_adaptation_log.csv", index=False)
print(f"  Saved -> depth_adaptation_log.csv")

current_acc = accuracy(params, X_test, y_test)
print(f"\n  Final Test Accuracy : {current_acc*100:.2f}%")
print(f"  Final Depths        : {current_depths}")

# ═══════════════════════════════════════════════════════════════
# 8.  HELPER: GATE COUNT
# ═══════════════════════════════════════════════════════════════
def get_gate_counts(depths, n_caps, cap_size):
    total_ry   = sum(depths) * cap_size
    total_rot  = sum(depths) * cap_size
    total_cnot = sum(
        (cap_size if cap_size > 2 else cap_size - 1) * (d // 2 + 1)
        for d in depths
    )
    total_cnot += (n_caps - 1) * (max(depths) // 2)
    return total_ry + total_rot + total_cnot

# ═══════════════════════════════════════════════════════════════
# 9.  EXPERIMENT 1 — EFFECTIVE DIMENSION
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 1] Computing Effective Dimension ...")

F       = qml.gradients.quantum_fisher(circuit)(params, X_test[0])
trace_F = float(np.trace(F))
F_hat   = (total_params * F) / trace_F if trace_F > 0 else F

ed_results = []
for n in N_SAMPLES_ED:
    scale    = n / (2 * np.pi * np.log(n))
    mat      = np.eye(total_params) + scale * F_hat
    sign, ld = np.linalg.slogdet(mat)
    eff_dim  = float((2 * ld) / np.log(scale)) if sign > 0 else float("nan")
    ed_results.append({"Sample_Size": n, "Effective_Dimension": eff_dim})
    print(f"  n={n:>6}  eff_dim={eff_dim:.4f}")

pd.DataFrame(ed_results).to_csv(
    f"{OUTPUT_DIR}effective_dimension.csv", index=False)
print(f"  Saved -> effective_dimension.csv")

# ═══════════════════════════════════════════════════════════════
# 10. EXPERIMENT 2 — EFFICIENCY ANALYSIS
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 2] Efficiency Analysis ...")

def train_quick(depths_cfg, n_ep=QUICK_EP):
    circ_tmp, tp, _ = build_circuit(N_QUBITS, N_CAPS, CAP_SIZE, depths_cfg, dev)
    _, mse_loss_tmp, accuracy_tmp = make_helpers(circ_tmp)

    np.random.seed(RANDOM_SEED)
    p   = np.random.uniform(0, np.pi, tp, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=LR)

    for _ in range(n_ep):
        perm = np.random.permutation(len(X_train))
        for s in range(0, len(X_train), BATCH_SIZE):
            Xb = X_train[perm[s: s + BATCH_SIZE]]
            yb = y_train[perm[s: s + BATCH_SIZE]]
            p, _ = opt.step_and_cost(
                lambda pp, _Xb=Xb, _yb=yb: mse_loss_tmp(pp, _Xb, _yb), p
            )

    return accuracy_tmp(p, X_test, y_test)

depths_d2 = [2] * N_CAPS
depths_d4 = [4] * N_CAPS

print(f"  Training Fixed_D2 {depths_d2} ({QUICK_EP} epochs) ...")
acc_d2 = train_quick(depths_d2)
print(f"  Fixed_D2  test_acc={acc_d2:.3f}")

print(f"  Training Fixed_D4 {depths_d4} ({QUICK_EP} epochs) ...")
acc_d4 = train_quick(depths_d4)
print(f"  Fixed_D4  test_acc={acc_d4:.3f}")

efficiency_data = [
    {
        "Model":      f"Adaptive_QCapsule {current_depths}",
        "Gate_Count": get_gate_counts(current_depths, N_CAPS, CAP_SIZE),
        "Accuracy":   round(current_acc, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D2 {depths_d2}",
        "Gate_Count": get_gate_counts(depths_d2,      N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d2, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D4 {depths_d4}",
        "Gate_Count": get_gate_counts(depths_d4,      N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d4, 4),
    },
]

pd.DataFrame(efficiency_data).to_csv(
    f"{OUTPUT_DIR}efficiency_analysis.csv", index=False)
print(f"  Saved -> efficiency_analysis.csv")

# ═══════════════════════════════════════════════════════════════
# 11. EXPERIMENT 3 — CAPSULE CORRELATION MATRIX
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 3] Capsule Correlation Matrix ...")

n_corr_samples  = min(50, len(X_test))
capsule_outputs = []
for x in X_test[:n_corr_samples]:
    out = circuit(params, x)
    capsule_outputs.append(np.array(out))

df_caps = pd.DataFrame(
    np.array(capsule_outputs),
    columns=[f"Capsule_{i}" for i in range(N_CAPS)]
)
corr_matrix = df_caps.corr()
corr_matrix.to_csv(f"{OUTPUT_DIR}capsule_correlation.csv")
print(f"  Saved -> capsule_correlation.csv")

# ═══════════════════════════════════════════════════════════════
# 12. DONE
# ═══════════════════════════════════════════════════════════════
print(f"""
All experiments complete.
  Digit pair        : {DIGIT_A} vs {DIGIT_B}
  Initial Depths    : {DEPTHS}
  Final Depths      : {current_depths}
  Final Test Acc    : {current_acc*100:.2f}%
  Files saved in    : {OUTPUT_DIR}
    training_log.csv           (Loss + Acc + Depths + Grad Variance)
    grad_variance_log.csv      (Epoch vs Gradient Variance)
    depth_adaptation_log.csv   (কোন epoch এ কোন capsule adapt হলো)
    effective_dimension.csv
    efficiency_analysis.csv
    capsule_correlation.csv
""")

  QCN Config
  Digits       : 0 vs 1
  Capsules     : 4  x  4 qubits  =  16 total
  Depths       : [4, 3, 3, 4]
  PCA shape    : 64  ->  16  ->  4x4 image
  Epochs       : 30   Batch: 8   LR: 0.05

[Data] Loading sklearn digits (0 vs 1) ...
  PCA variance explained : 93.5%
  Train : 270   Test : 90
  Input per sample       : flat 16 = 4x4 compressed image

Circuit ready  |  total_params = 224

[Train] Starting training (30 epochs) ...
  Epoch   1/30  |  loss=0.8283  train_acc=0.930  test_acc=0.867  grad_var=4.609e-03


KeyboardInterrupt: 

In [2]:
import subprocess, sys

packages = [
    "pennylane", "cirq", "qualtran", "pennylane-cirq", "autoray"
]
subprocess.run([sys.executable, "-m", "pip", "uninstall"] + packages + ["-y"])

# PennyLane 0.38 এর compatible stack
subprocess.run([sys.executable, "-m", "pip", "install",
    "pennylane==0.38.0",
    "autoray==0.6.7",
    "autograd==1.6.2",
])

CompletedProcess(args=['c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', '-m', 'pip', 'install', 'pennylane==0.38.0', 'autoray==0.6.7', 'autograd==1.6.2'], returncode=1)

In [3]:
import pennylane as qml
print(qml.__version__)  # 0.38.0

ModuleNotFoundError: No module named 'pennylane'

In [1]:
"""
Adaptive Quantum Capsule Network (QCN) — sklearn Digits Binary Classification
==============================================================================
FIX: Pin PennyLane to 0.38.0 to avoid cirq/qualtran import error.
     Install: pip install "pennylane==0.38.0"
              pip uninstall cirq qualtran -y   (remove if installed)

Architecture : N_CAPS capsules × CAP_SIZE qubits/capsule
Preprocessing:
    Raw image  : 8×8 = 64 pixels
    PCA        : 64  →  N_QUBITS components   (N_QUBITS = N_CAPS × CAP_SIZE)
    Reshape    : N_QUBITS  →  IMG_SIDE × IMG_SIDE  "compressed image"
    Scale      : MinMaxScaler  →  [0, π]  (angle encoding into qubits)

Outputs:
  training_log.csv           ← Epoch, Loss, Train_Acc, Test_Acc, Grad_Var_Mean, Grad_Var_Std
  grad_variance_log.csv      ← Epoch vs Gradient Variance (dedicated CSV)
  effective_dimension.csv
  efficiency_analysis.csv
  capsule_correlation.csv
"""

import os
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ═══════════════════════════════════════════════════════════════
# 0.  ALL HYPERPARAMETERS — change only here, everything derives
# ═══════════════════════════════════════════════════════════════

DIGIT_A     = 0              # first digit class
DIGIT_B     = 1              # second digit class

N_CAPS      = 4              # number of capsules
CAP_SIZE    = 4              # qubits per capsule
DEPTHS      = [4, 3, 3, 4]  # per-capsule circuit depth  (len must == N_CAPS)

N_EPOCHS    = 30             # training epochs
BATCH_SIZE  = 8              # mini-batch size
LR          = 0.05           # Adam learning rate

QUICK_EP    = 5              # epochs for baseline models in efficiency analysis
TEST_SIZE   = 0.25           # fraction of data held out for testing
RANDOM_SEED = 42

OUTPUT_DIR  = "C:\\Users\\User\\Downloads\\Barren-Pleatau-Model\\"

# ── Derived constants (do NOT edit below this line) ─────────────
N_QUBITS     = N_CAPS * CAP_SIZE
IMG_SIDE     = int(np.sqrt(N_QUBITS))
N_SAMPLES_ED = [100, 500, 1000, 5000, 10000]

assert len(DEPTHS) == N_CAPS,     "len(DEPTHS) must equal N_CAPS"
assert IMG_SIDE ** 2 == N_QUBITS, "N_QUBITS must be a perfect square"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 60)
print(f"  QCN Config")
print(f"  Digits       : {DIGIT_A} vs {DIGIT_B}")
print(f"  Capsules     : {N_CAPS}  x  {CAP_SIZE} qubits  =  {N_QUBITS} total")
print(f"  Depths       : {DEPTHS}")
print(f"  PCA shape    : 64  ->  {N_QUBITS}  ->  {IMG_SIDE}x{IMG_SIDE} image")
print(f"  Epochs       : {N_EPOCHS}   Batch: {BATCH_SIZE}   LR: {LR}")
print("=" * 60)

# ═══════════════════════════════════════════════════════════════
# 1.  DATASET  +  PCA PREPROCESSING
# ═══════════════════════════════════════════════════════════════
print(f"\n[Data] Loading sklearn digits ({DIGIT_A} vs {DIGIT_B}) ...")

digits = load_digits()
mask   = (digits.target == DIGIT_A) | (digits.target == DIGIT_B)
X_raw  = digits.data[mask]
y_raw  = digits.target[mask]

y = np.where(y_raw == DIGIT_A, -1.0, 1.0)

pca      = PCA(n_components=N_QUBITS, random_state=RANDOM_SEED)
X_pca    = pca.fit_transform(X_raw)

scaler   = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

X_images = X_scaled.reshape(-1, IMG_SIDE, IMG_SIDE)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f"  PCA variance explained : {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"  Train : {len(X_train)}   Test : {len(X_test)}")
print(f"  Input per sample       : flat {N_QUBITS} = {IMG_SIDE}x{IMG_SIDE} compressed image")

# ═══════════════════════════════════════════════════════════════
# 2.  QUANTUM DEVICE
# ═══════════════════════════════════════════════════════════════
dev = qml.device("default.qubit", wires=N_QUBITS)

# ═══════════════════════════════════════════════════════════════
# 3.  CIRCUIT BUILDER
# ═══════════════════════════════════════════════════════════════
def build_circuit(n_qubits, n_caps, cap_size, depths, device):
    offsets = [0]
    for c in range(n_caps):
        offsets.append(offsets[-1] + depths[c] * cap_size * 4)
    total_params = offsets[-1]
    max_d        = max(depths)

    @qml.qnode(device, diff_method="best")
    def circuit(params, x):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)

        for layer in range(max_d):
            for c in range(n_caps):
                if layer >= depths[c]:
                    continue

                w  = list(range(c * cap_size, (c + 1) * cap_size))
                cp = params[offsets[c]: offsets[c + 1]].reshape(
                         depths[c], cap_size, 4)

                xc = [float(x[c * cap_size + j]) for j in range(cap_size)]

                for j, wire in enumerate(w):
                    qml.RY(xc[j] * cp[layer, j, 3], wires=wire)

                for j, wire in enumerate(w):
                    qml.Rot(cp[layer, j, 0],
                            cp[layer, j, 1],
                            cp[layer, j, 2], wires=wire)

                if layer % 2 == 0:
                    for j in range(len(w) - 1):
                        qml.CNOT(wires=[w[j], w[j + 1]])
                    if len(w) > 2:
                        qml.CNOT(wires=[w[-1], w[0]])

            if layer % 2 == 0:
                ic_offset = (layer // 2) % 2
                for c in range(ic_offset, n_caps - 1, 2):
                    if layer < depths[c] and layer < depths[c + 1]:
                        boundary_a = (c + 1) * cap_size - 1
                        boundary_b = (c + 1) * cap_size
                        qml.CNOT(wires=[boundary_a, boundary_b])

        return qml.math.stack([
            qml.expval(qml.PauliZ(c * cap_size)) for c in range(n_caps)
        ])

    return circuit, total_params, offsets


circuit, total_params, offsets = build_circuit(
    N_QUBITS, N_CAPS, CAP_SIZE, DEPTHS, dev
)
print(f"\nCircuit ready  |  total_params = {total_params}")

# ═══════════════════════════════════════════════════════════════
# 4.  LOSS + PREDICTION HELPERS
# ═══════════════════════════════════════════════════════════════
def predict_one(params, x):
    return qml.math.mean(circuit(params, x))


def mse_loss(params, X_batch, y_batch):
    preds = qml.math.stack([predict_one(params, x) for x in X_batch])
    return qml.math.mean((preds - y_batch) ** 2)


def accuracy(params, X, y):
    signs = np.array([float(np.sign(predict_one(params, x))) for x in X])
    signs = np.where(signs == 0, 1.0, signs)
    return float(np.mean(signs == y))

# ═══════════════════════════════════════════════════════════════
# 5.  TRAINING LOOP  +  GRADIENT VARIANCE TRACKING
# ═══════════════════════════════════════════════════════════════
print(f"\n[Train] Starting training ({N_EPOCHS} epochs) ...")

np.random.seed(RANDOM_SEED)
params    = np.random.uniform(0, np.pi, total_params, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=LR)

# autograd gradient function
grad_fn = qml.grad(mse_loss)

training_log   = []
grad_var_log   = []

for epoch in range(1, N_EPOCHS + 1):
    perm   = np.random.permutation(len(X_train))
    X_shuf = X_train[perm]
    y_shuf = y_train[perm]

    epoch_loss, n_batches = 0.0, 0

    for start in range(0, len(X_train), BATCH_SIZE):
        Xb = X_shuf[start: start + BATCH_SIZE]
        yb = y_shuf[start: start + BATCH_SIZE]
        params, loss_val = optimizer.step_and_cost(
            lambda p: mse_loss(p, Xb, yb), params
        )
        epoch_loss += float(loss_val)
        n_batches  += 1

    avg_loss  = epoch_loss / n_batches
    train_acc = accuracy(params, X_train, y_train)
    test_acc  = accuracy(params, X_test,  y_test)

    # ── Gradient Variance across 16 random training samples ──────
    grad_samples = []
    eval_indices = perm[:min(16, len(X_train))]

    for idx in eval_indices:
        g = grad_fn(params,
                    X_train[idx: idx + 1],
                    y_train[idx: idx + 1])
        grad_samples.append(np.array(g[0]).flatten())

    grad_samples        = np.array(grad_samples)
    param_variances     = np.var(grad_samples, axis=0)
    epoch_grad_var_mean = float(np.mean(param_variances))
    epoch_grad_var_std  = float(np.std(param_variances))

    training_log.append({
        "Epoch":         epoch,
        "Loss":          round(avg_loss,  6),
        "Train_Acc":     round(train_acc, 4),
        "Test_Acc":      round(test_acc,  4),
        "Grad_Var_Mean": f"{epoch_grad_var_mean:.4e}",
        "Grad_Var_Std":  f"{epoch_grad_var_std:.4e}",
    })

    grad_var_log.append({
        "Epoch":         epoch,
        "Grad_Var_Mean": epoch_grad_var_mean,
        "Grad_Var_Std":  epoch_grad_var_std,
    })

    print(f"  Epoch {epoch:>3}/{N_EPOCHS}  |  "
          f"loss={avg_loss:.4f}  "
          f"train_acc={train_acc:.3f}  "
          f"test_acc={test_acc:.3f}  "
          f"grad_var={epoch_grad_var_mean:.3e}")

# ── Save CSVs ────────────────────────────────────────────────────
pd.DataFrame(training_log).to_csv(
    f"{OUTPUT_DIR}training_log.csv", index=False)
print(f"\n  Saved -> training_log.csv")

pd.DataFrame(grad_var_log).to_csv(
    f"{OUTPUT_DIR}grad_variance_log.csv", index=False)
print(f"  Saved -> grad_variance_log.csv  (Epoch vs Gradient Variance)")

current_acc = accuracy(params, X_test, y_test)
print(f"\n  Final Test Accuracy : {current_acc*100:.2f}%")

# ═══════════════════════════════════════════════════════════════
# 6.  HELPER: GATE COUNT
# ═══════════════════════════════════════════════════════════════
def get_gate_counts(depths, n_caps, cap_size):
    total_ry   = sum(depths) * cap_size
    total_rot  = sum(depths) * cap_size
    total_cnot = sum(
        (cap_size if cap_size > 2 else cap_size - 1) * (d // 2 + 1)
        for d in depths
    )
    total_cnot += (n_caps - 1) * (max(depths) // 2)
    return total_ry + total_rot + total_cnot

# ═══════════════════════════════════════════════════════════════
# 7.  EXPERIMENT 1 — EFFECTIVE DIMENSION
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 1] Computing Effective Dimension ...")

F       = qml.gradients.quantum_fisher(circuit)(params, X_test[0])
trace_F = float(np.trace(F))
F_hat   = (total_params * F) / trace_F if trace_F > 0 else F

ed_results = []
for n in N_SAMPLES_ED:
    scale    = n / (2 * np.pi * np.log(n))
    mat      = np.eye(total_params) + scale * F_hat
    sign, ld = np.linalg.slogdet(mat)
    eff_dim  = float((2 * ld) / np.log(scale)) if sign > 0 else float("nan")
    ed_results.append({"Sample_Size": n, "Effective_Dimension": eff_dim})
    print(f"  n={n:>6}  eff_dim={eff_dim:.4f}")

pd.DataFrame(ed_results).to_csv(
    f"{OUTPUT_DIR}effective_dimension.csv", index=False)
print(f"  Saved -> effective_dimension.csv")

# ═══════════════════════════════════════════════════════════════
# 8.  EXPERIMENT 2 — EFFICIENCY ANALYSIS
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 2] Efficiency Analysis ...")

def train_quick(depths_cfg, n_ep=QUICK_EP):
    circ_tmp, tp, _ = build_circuit(N_QUBITS, N_CAPS, CAP_SIZE, depths_cfg, dev)

    def predict_one_tmp(p, x):
        return qml.math.mean(circ_tmp(p, x))

    def mse_loss_tmp(p, X_batch, y_batch):
        preds = qml.math.stack([predict_one_tmp(p, x) for x in X_batch])
        return qml.math.mean((preds - y_batch) ** 2)

    def accuracy_tmp(p, X, y):
        signs = np.array([float(np.sign(predict_one_tmp(p, x))) for x in X])
        signs = np.where(signs == 0, 1.0, signs)
        return float(np.mean(signs == y))

    np.random.seed(RANDOM_SEED)
    p   = np.random.uniform(0, np.pi, tp, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=LR)

    for _ in range(n_ep):
        perm = np.random.permutation(len(X_train))
        for s in range(0, len(X_train), BATCH_SIZE):
            Xb = X_train[perm[s: s + BATCH_SIZE]]
            yb = y_train[perm[s: s + BATCH_SIZE]]
            p, _ = opt.step_and_cost(
                lambda pp, _Xb=Xb, _yb=yb: mse_loss_tmp(pp, _Xb, _yb), p
            )

    return accuracy_tmp(p, X_test, y_test)

depths_d2 = [2] * N_CAPS
depths_d4 = [4] * N_CAPS

print(f"  Training Fixed_D2 {depths_d2} ({QUICK_EP} epochs) ...")
acc_d2 = train_quick(depths_d2)
print(f"  Fixed_D2  test_acc={acc_d2:.3f}")

print(f"  Training Fixed_D4 {depths_d4} ({QUICK_EP} epochs) ...")
acc_d4 = train_quick(depths_d4)
print(f"  Fixed_D4  test_acc={acc_d4:.3f}")

efficiency_data = [
    {
        "Model":      f"Adaptive_QCapsule {DEPTHS}",
        "Gate_Count": get_gate_counts(DEPTHS,    N_CAPS, CAP_SIZE),
        "Accuracy":   round(current_acc, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D2 {depths_d2}",
        "Gate_Count": get_gate_counts(depths_d2, N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d2, 4),
    },
    {
        "Model":      f"Fixed_QCapsule_D4 {depths_d4}",
        "Gate_Count": get_gate_counts(depths_d4, N_CAPS, CAP_SIZE),
        "Accuracy":   round(acc_d4, 4),
    },
]

pd.DataFrame(efficiency_data).to_csv(
    f"{OUTPUT_DIR}efficiency_analysis.csv", index=False)
print(f"  Saved -> efficiency_analysis.csv")

# ═══════════════════════════════════════════════════════════════
# 9.  EXPERIMENT 3 — CAPSULE CORRELATION MATRIX
# ═══════════════════════════════════════════════════════════════
print("\n[Exp 3] Capsule Correlation Matrix ...")

n_corr_samples  = min(50, len(X_test))
capsule_outputs = []
for x in X_test[:n_corr_samples]:
    out = circuit(params, x)
    capsule_outputs.append(np.array(out))

df_caps = pd.DataFrame(
    np.array(capsule_outputs),
    columns=[f"Capsule_{i}" for i in range(N_CAPS)]
)
corr_matrix = df_caps.corr()
corr_matrix.to_csv(f"{OUTPUT_DIR}capsule_correlation.csv")
print(f"  Saved -> capsule_correlation.csv")

# ═══════════════════════════════════════════════════════════════
# 10. DONE
# ═══════════════════════════════════════════════════════════════
print(f"""
All experiments complete.
  Digit pair        : {DIGIT_A} vs {DIGIT_B}
  PCA components    : {N_QUBITS}  ->  {IMG_SIDE}x{IMG_SIDE} compressed image
  Final Test Acc    : {current_acc*100:.2f}%
  Files saved in    : {OUTPUT_DIR}
    training_log.csv         (Loss + Acc + Grad Variance per epoch)
    grad_variance_log.csv    (Epoch vs Gradient Variance — dedicated)
    effective_dimension.csv
    efficiency_analysis.csv
    capsule_correlation.csv
""")

  QCN Config
  Digits       : 0 vs 1
  Capsules     : 4  x  4 qubits  =  16 total
  Depths       : [4, 3, 3, 4]
  PCA shape    : 64  ->  16  ->  4x4 image
  Epochs       : 30   Batch: 8   LR: 0.05

[Data] Loading sklearn digits (0 vs 1) ...
  PCA variance explained : 93.5%
  Train : 270   Test : 90
  Input per sample       : flat 16 = 4x4 compressed image

Circuit ready  |  total_params = 224

[Train] Starting training (30 epochs) ...
  Epoch   1/30  |  loss=0.8283  train_acc=0.930  test_acc=0.867  grad_var=4.609e-03
  Epoch   2/30  |  loss=0.6914  train_acc=0.933  test_acc=0.922  grad_var=4.001e-03
  Epoch   3/30  |  loss=0.6398  train_acc=0.952  test_acc=0.933  grad_var=4.584e-03
  Epoch   4/30  |  loss=0.5474  train_acc=0.970  test_acc=0.933  grad_var=4.055e-03
  Epoch   5/30  |  loss=0.5304  train_acc=0.948  test_acc=0.933  grad_var=4.030e-03
  Epoch   6/30  |  loss=0.5090  train_acc=0.952  test_acc=0.911  grad_var=4.066e-03
  Epoch   7/30  |  loss=0.4947  train_acc=0.952  test_acc